In [ ]:
import os

import math
import random
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset as Dataset1
#from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
# ※モデル構造と重み、トークナイザのみ公式リポジトリから取得するために使用します
#from transformers import T5ForConditionalGeneration, T5Tokenizer, DataCollatorForSeq2Seq, T5EncoderModel
#deimport levenshtein
from nltk import bleu_score
import datetime
from transformers import logging
import matplotlib.pyplot as plt
import seaborn as sns
#from transformers import BertTokenizer, BertModel, BertForPreTraining, DataCollatorForSeq2Seq
import pandas as pd
#from datasets import Dataset
from transformers import BertTokenizer, BertModel, DataCollatorWithPadding
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from evaluate import load
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.bleu_score import sentence_bleu
from tqdm.notebook import tqdm
import copy
from collections import deque, OrderedDict
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, Subset
from transformers import get_cosine_schedule_with_warmup
from torch.amp import autocast, GradScaler

In [ ]:
#device = torch.device( "cuda:0" )
model_checkpoint = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_checkpoint, model_max_length=512)

pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id
mask_token_id = tokenizer.mask_token_id
## 2. 新しい特殊トークンを登録
special_tokens_dict = {'additional_special_tokens': ['[unused0]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
special_tokens_dict = {'additional_special_tokens': ['[unused1]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
sos_token_id = tokenizer.encode( "[unused0]" )[1]
print( "sos_token_id:", sos_token_id )
eos_token_id = tokenizer.encode( "[unused1]" )[1]
print( "eos_token_id:", eos_token_id )
a_token_id = tokenizer.encode( "a"  )[1]
print( "a_token_id:", a_token_id )
an_token_id = tokenizer.encode( "an"  )[1]
the_token_id = tokenizer.encode( "the" )[1]
and_token_id = tokenizer.encode( "and" )[1]
in_token_id = tokenizer.encode( "in" )[1]
of_token_id = tokenizer.encode( "of" )[1]
on_token_id = tokenizer.encode( "on" )[1]
at_token_id = tokenizer.encode( "at" )[1]
to_token_id = tokenizer.encode( "to" )[1]
for_token_id = tokenizer.encode( "for" )[1]
from_token_id = tokenizer.encode( "from" )[1]
with_token_id = tokenizer.encode( "with" )[1]
by_token_id = tokenizer.encode( "by" )[1]
we_token_id = tokenizer.encode( "we" )[1]
i_token_id = tokenizer.encode( "i" )[1]
you_token_id = tokenizer.encode( "you" )[1]
he_token_id = tokenizer.encode( "he" )[1]
she_token_id = tokenizer.encode( "she" )[1]
it_token_id = tokenizer.encode( "it" )[1]
they_token_id = tokenizer.encode( "they" )[1]
period_token_id = tokenizer.encode( "." )[1]
comma_token_id = tokenizer.encode( "," )[1]
dbl_token_id = tokenizer.encode( '"' )[1]
sgl_token_id = tokenizer.encode( "'" )[1]

# 辞書サイズを保存
vocab_size = len( tokenizer )

print( "vocab_size:", vocab_size )


def logsumexp(x, dim=1):
    return torch.logsumexp(x.float(), dim=dim).type_as(x)

class DynamicCRF(nn.Module):
    
    def __init__(self, num_embedding, low_rank=32, beam_size=64, crf_coef=1.0, temp = 1.0, top_p = 0.9, top_k = 50, cand = 0,
                 num_samples = 16, sft_mode = False ):
        super().__init__()

        self.E1 = nn.Embedding(num_embedding, low_rank)
        self.E2 = nn.Embedding(num_embedding, low_rank)

        self.vocab_size = num_embedding
        self.rank = low_rank
        self.beam = beam_size
        self.crf_coef = crf_coef
        self.temp = temp
        self.top_p = top_p
        self.top_k = top_k
        self.num_samples = num_samples
        self.cand = cand
        self.sft_mode = sft_mode
    
    def _compute_score(self, emissions, targets, masks=None):
        batch_size, seq_len = targets.size()
        emission_scores = emissions.gather(2, targets[:, :, None])[:, :, 0]  # B x T
        transition_scores = (self.E1(targets[:, :-1]) * self.E2(targets[:, 1:])).sum(2)

        scores = emission_scores
        scores[:, 1:] += transition_scores

        if masks is not None:
            scores = scores * masks.type_as(scores)

        return scores.sum(-1)

    def _compute_normalizer(self, emissions, targets=None, masks=None, beam=None):
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        if targets is not None:
            #_emissions = emissions.scatter(2, targets[:, :, None], np.float('inf'))
            _emissions = emissions.scatter(2, targets[:, :, None], float('inf'))
            beam_targets = _emissions.topk(beam, 2)[1]
            beam_emission_scores = emissions.gather(2, beam_targets)
        else:
            beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D; position i - 1, previous step.
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D; position i, current step.
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam)

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        for i in range(1, seq_len):
            next_score = score[:, :, None] + beam_transition_matrix[:, i-1]
            next_score = logsumexp(next_score, dim=1) + beam_emission_scores[:, i]

            if masks is not None:
                score = torch.where(masks[:, i:i+1], next_score, score)
            else:
                score = next_score

        # Sum (log-sum-exp) over all possible tags
        return logsumexp(score, dim=1)


    def _compute_grpo_samples( self, beam_emission_scores, beam_transition_matrix, beam_targets, sampled_beam_idx=None, \
                                repeat_penalty = 0.0001, targets=None, masks=None, beam=None):

        eps = 1e-8
        device = beam_emission_scores.device

        permit_repeat = [ pad_token_id, eos_token_id, cls_token_id, sep_token_id, a_token_id, an_token_id, the_token_id, period_token_id, \
                         comma_token_id, and_token_id, in_token_id, we_token_id, i_token_id, you_token_id, he_token_id, she_token_id, \
                         it_token_id, they_token_id, dbl_token_id, sgl_token_id ]
        permit_repeat = torch.tensor( permit_repeat ).to(device)

        beam = beam if beam is not None else self.beam
        batch_size, seq_len, beam= beam_emission_scores.size()

        if sampled_beam_idx is not None:
            sampled_beam_idx_flag = True #sample_beam_idx があったら true
        else:
            sampled_beam_idx_flag = False #sample_beam_idx がなかったら false

        # フィルタリング用のパラメータ設定 (config等から取得できるよう適宜調整してください)
        #top_k = 50  # 上位k個に絞る (0なら無効)
        #top_p = 0.9 # 累積確率pまでに絞る (1.0なら無効)

        traj_tokens = []
        finalized_tokens = []
        step_probs = []

        score = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1)
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()
        #dummy2 = dummy[:,:, None ].expand( -1, -1, self.num_samples )
        if masks is not None:
            masks = masks[:,:,None].expand(-1, -1, self.num_samples ).permute( 0,2,1)
        B, N, C, W = score.unsqueeze(-1).expand(-1,-1,-1,beam).shape

        logits_t0 = score / self.temp

        beam_history = beam_targets[:,None, 0].expand( -1, self.num_samples, -1 ).unsqueeze(-1) # [B, num_samples,  beam, 1]

        for i in range(1, seq_len):

            _score_matrix = score.unsqueeze(-1) + beam_transition_matrix[:,i-1,None,:,:,].expand( -1, N, -1, -1 )
            _score_matrix = _score_matrix + beam_emission_scores[:,i][:,None,None,:].expand(-1,N,C,-1)

            # -----------------------------------------------------------------
            # ★ リピートペナルティの動的適用
            # -----------------------------------------------------------------

            if repeat_penalty > 0.0:
                # to_tokens shape: [B, N, 1, beam_to]
                to_tokens = beam_targets[:, i].unsqueeze(1)[:,None].expand( -1, self.num_samples, -1, -1)

                # is_permitted shape: [B, 1, beam_to]
                is_permitted = torch.isin(to_tokens, permit_repeat)

                # history shape: [B, N, beam_from, past length, 1]
                # target  shape: [B, N, 1,         1,           beam_to]
                history_expanded = beam_history.unsqueeze(-1)
                to_tokens_expanded = to_tokens.unsqueeze(3)

                # match_matrix shape: [B, N, beam_from, past length, beam_to]
                match_matrix = (history_expanded == to_tokens_expanded)

                # repeat_mask shape: [B, beam_from, beam_to]
                repeat_mask = match_matrix.any(dim=3) & (~is_permitted)
                #repeat_count = torch.sum( match_matrix.float(), dim = 2 )
                #repeat_count = repeat_count * is_permitted.float()

                #log_penalty = torch.log( torch.tensor( repeat_penalty))
                #penalty_matrix = log_penalty * repeat_count
                #penalty_delta = torch.where( repeat_count != 0.0, penalty_matrix, 0.0 )
                penalty_matrix = repeat_mask.float()
                log_penalty = torch.log( torch.tensor( repeat_penalty))
                penalty_delta = torch.where( penalty_matrix == 1.0, log_penalty, 0.0)
                #penalty_delta = penalty_delta[:,None,:,:].expand( -1, N, -1, -1 )
                _search_score_matrix = _score_matrix + penalty_delta
            else:
                _search_score_matrix = _score_matrix

           #step_probs.append( _search_score_matrix / self.temp ) # (S), B, N, C, W

            ## 通常のビタビ処理（ペナルティ適用後のスコア行列から MAX を取る）
            #_score, _index = _search_score_matrix.max(dim=1) # bsz, beam

            B, N, C, W = _search_score_matrix.shape
            flat_score = _search_score_matrix.permute(0, 1, 3, 2 ).reshape(-1, C)

            step_probs.append(_search_score_matrix / self.temp)

            # --- Top-K / Top-P Filtering 開始 (修正版) ---
            logits = flat_score / self.temp #B*W*N,C

            # 1. まず Top-K で上位K個に絞る (0なら無効)
            if self.top_k > 0:
                top_k_val = min(self.top_k, logits.size(-1)) # top_kが語彙数より大きくならないように調整
                top_k_logits, top_k_indices = torch.topk(logits, top_k_val, dim=-1)

                # Top-K用のマスクを作成
                min_values = top_k_logits[:, -1].unsqueeze(-1)
                logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)

                # 以降の処理（Top-P）のために top_k_logits, top_k_indices を更新
                # （注: top_pと組み合わせる場合、ここでのlogitsの更新より、
                #  後続のtop_k_indicesを使ったmaskの方がロジックが整合しやすい）

            # Top-Kの変数を再定義（top_pの処理で使うため）
            # top_k > 0 の場合、top_kで絞った後の値を使う。0の場合は全範囲。
            top_k_logits, top_k_indices = logits, torch.arange(logits.size(-1), device=logits.device).expand(logits.size(0), -1)
            # ↑ このアプローチはメモリを食うため、元の実装の通りtop_k_indicesでmaskする方が綺麗です。
            # 以下、元のロジックを活かした修正版です。
            # 2. Top-P (Nucleus) filtering
            if self.top_p < 1.0:
                # Top-K/Allで絞ったテンソルでソート
                sorted_logits, sorted_indices = torch.sort(top_k_logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                # 除去対象のマスクを作成
                sorted_indices_to_remove = cumulative_probs > self.top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                # top_k_logitsと同じ形状のマスクを作成
                indices_to_remove_k = torch.zeros_like(top_k_logits, dtype=torch.bool).scatter_(
                    dim=-1, index=sorted_indices, src=sorted_indices_to_remove
                )
                top_k_logits[indices_to_remove_k] = -float('Inf')

            # 3. 全語彙の logits を一旦すべて -inf にして、生き残った top_k_logits だけを戻す
            new_logits = torch.full_like(logits, float('-inf'))
            new_logits.scatter_(dim=-1, index=top_k_indices, src=top_k_logits)
            logits = new_logits

            if torch.all(logits == float('inf')):
                logits = torch.ones_like(logits)  # テンソル全体を1にする場合

            probs = F.softmax(logits, dim=-1) # B*W*N,C この softmax は、C についての softmax

            #torch.manual_seed(42)
            _index_flat = torch.multinomial(probs, num_samples=1, replacement=False) #B*N*W
            #_index_flat = torch.topk( probs, self.cand, dim = -1 )
            #print( "_index_flat.siez():", _index_flat.size() )
            #_index_flat1 = _index_flat[:,:1]

            _score_flat = torch.gather( flat_score, -1, _index_flat)
            _index2 = _index_flat.view(B, N, W) # B, N, W
            _score = _score_flat.view(B, N, W) # B, N, W

            if masks is not None:
                score = torch.where(masks[:, :, i: i+1], _score, score)
                index2 = torch.where(masks[:, :,  i: i+1], _index2, dummy )
            else:
                score, index2 = _score, _index2

            #traj_tokens.append(index)

            history_len = beam_history.size(-1)
            gather_index2 = index2.unsqueeze(-1).expand( -1, -1, -1, history_len )
            updated_history = torch.gather(beam_history, 2, gather_index2)

            current_tokens = beam_targets[:, i].unsqueeze(-1)[:,None].expand( -1, self.num_samples, -1, -1 )
            beam_history = torch.cat([updated_history, current_tokens], dim=-1)


            traj_tokens.append( index2 ) # S, B, N, W

        ## 2. サンプリングされた最後のインデックスを取得 (B, N, 1)
        B, N, C = score.shape
        flat_score = score.reshape(-1, C)

        probs = F.softmax(flat_score / self.temp, dim=-1)
        _index_flat = torch.multinomial(probs, num_samples=1, replacement = False ) #B*N

        _index = _index_flat.view(B, N)
        current_sampled_index = _index #(B, N )

        ## 3. 最初の要素として追加
        finalized_tokens.append( current_sampled_index.unsqueeze(-1) ) # (S), B, N, 1

        # バックトレーシング
        #if not sampled_beam_idx_flag:
        #    beam_logits = []
        for i3, (idx, step_prob) in enumerate( zip( reversed(traj_tokens), reversed( step_probs ))):
            #traj_tokens #(S), B, N, W
            #step_probs (S), B, N, C, W
            previous_index = finalized_tokens[-1] # B, N
            # idx B, N, W
            # previous_index B, N, 1
            finalized_tokens.append(idx.gather(2, previous_index)) # (S), B, N

        finalized_tokens = torch.stack( finalized_tokens, dim = 0 )
        finalized_tokens = finalized_tokens.squeeze( -1 )
        finalized_tokens = torch.flip( finalized_tokens, dims = (0, ) )
        finalized_tokens = finalized_tokens.permute( 1, 0, 2 )
        if sampled_beam_idx_flag:
            sampled_beam_idx = sampled_beam_idx
        else:
            sampled_beam_idx = finalized_tokens
        finalized_tokens = beam_targets[:,:,None].expand(-1,-1,N,-1).gather(3, finalized_tokens[:, :, None])[:, :, 0]

        beam_logits = []
        for i3, step_prob in enumerate(reversed( step_probs )):
            i4 = seq_len - i3 - 1
            previous_beam_index = sampled_beam_idx[:,i4-1]
            prob_from_prev = step_prob.gather(2, previous_beam_index[:, :, None, None].expand(-1, -1, 1,  beam))
            beam_logits.append( prob_from_prev.squeeze(2) ) # (S), B, W

        beam_logits.append( logits_t0 )
        beam_logits = torch.stack( beam_logits, dim = 0 )
        beam_logits = torch.flip( beam_logits, dims = (0, ) )
        beam_logits = beam_logits.permute( 1, 0, 3, 2 ) # B,S,W,N
        beam_probs = F.softmax( beam_logits, dim = 2 )

        log_beam_probs = torch.log( beam_probs + eps )

        return beam_probs, log_beam_probs, sampled_beam_idx, finalized_tokens

    def _compute_grpo_samples_suppression( self, beam_emission_scores, beam_transition_matrix, beam_targets, sampled_beam_idx=None, \
                          targets=None, masks=None, beam=None):

        eps = 1e-8
        device = beam_emission_scores.device

        permit_repeat = [ pad_token_id, eos_token_id, cls_token_id, sep_token_id, a_token_id, an_token_id, the_token_id, period_token_id, \
                         comma_token_id, and_token_id, in_token_id, we_token_id, i_token_id, you_token_id, he_token_id, she_token_id, \
                         it_token_id, they_token_id, dbl_token_id, sgl_token_id ]

        beam = beam if beam is not None else self.beam
        batch_size, seq_len, beam= beam_emission_scores.size()

        if sampled_beam_idx is not None:
            sampled_beam_idx_flag = True
        else:
            sampled_beam_idx_flag = False

        # フィルタリング用のパラメータ設定 (config等から取得できるよう適宜調整してください)
        #top_k = 50  # 上位k個に絞る (0なら無効)
        #top_p = 0.9 # 累積確率pまでに絞る (1.0なら無効)

        traj_tokens = []
        step_probs = []

        score = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1)
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()
        dummy2 = dummy[:,:, :, None ].expand( -1, -1, -1, self.cand )
        #print( "dummy.size():", dummy.size() )
        #print( "dummy2.size():", dummy2.size() )
        if masks is not None:
            #print( "masks.size():", masks.size() )
            masks = masks[:,:,None].expand(-1, -1, self.num_samples ).permute( 0,2,1)
            #print( "masks.size():", masks.size() )
        #print( "1 score:", score )
        B, N, C, W = score.unsqueeze(-1).expand(-1,-1,-1,beam).shape
        #flat_score = score.unsqueeze(-1).expand(-1,-1,-1,beam).permute(0, 1, 3, 2).reshape(-1, C)
        #logits = flat_score / self.temp # B*N*W*cand, C
        #probs = F.softmax(logits, dim=-1)  # B*N*W*cand,C この softmax は、C についての softmax

        #torch.manual_seed(42)
        #_index_flat = torch.multinomial(probs, num_samples=self.cand, replacement=False)
        #_, _index_flat = torch.topk( probs, self,cand, -1)

        #score2 = torch.gather( probs, -1, _index_flat).view( B, N, W, self.cand)
        #score2 = torch.gather( flat_score, -1, _index_flat).view( B, N, W, self.cand)
        #score2 = torch.gather( flat_score / self.temp, -1, _index_flat).view( B, N, W, self.cand)
        #log_probs_t0 = torch.log_softmax( score / self.temp, dim=-1)
        logits_t0 = score / self.temp
        #score2 = score[:,:,:,None].expand(-1,-1,-1, self.cand)

        for i in range(1, seq_len):
            #_score = score.unsqueeze(-1) + beam_transition_matrix[:, i-1,None,:,:].expand( -1, N,-1,-1) #B,N,C,W

            _score_matrix = score.unsqueeze(-1) + beam_transition_matrix[:,i-1,None,:,:,].expand( -1, N, -1, -1 )
            _score_matrix = _score_matrix + beam_emission_scores[:,i][:,None,None,:].expand(-1,N,C,-1)

            # 【重要】強化学習用の遷移対数確率（次のBeam候補 dim=-1 に対する確率分布）
            # 形状: (bsz, beam, beam)
            #action_log_prob = torch.log_softmax(_score_matrix / self.temp, dim=-1)
            ##action_log_prob = _score_matrix
            ##action_log_prob = _score_matrix / self.temp
            #step_log_probs.append(action_log_prob)
            step_probs.append( _score_matrix / self.temp )

            #_score, _index = _score_matrix.max( dim = 1 )

            B, N, C, W = _score_matrix.shape
            flat_score = _score_matrix.permute(0, 1, 3, 2 ).reshape(-1, C)

            # --- Top-K / Top-P Filtering 開始 (修正版) ---
            logits = flat_score / self.temp #B*W*N,C

            # 1. まず Top-K で上位K個に絞る (0なら無効)
            if self.top_k > 0:
                top_k_val = min(self.top_k, logits.size(-1)) # top_kが語彙数より大きくならないように調整
                top_k_logits, top_k_indices = torch.topk(logits, top_k_val, dim=-1)

                # Top-K用のマスクを作成
                min_values = top_k_logits[:, -1].unsqueeze(-1)
                logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)

                # 以降の処理（Top-P）のために top_k_logits, top_k_indices を更新
                # （注: top_pと組み合わせる場合、ここでのlogitsの更新より、
                #  後続のtop_k_indicesを使ったmaskの方がロジックが整合しやすい）

            # Top-Kの変数を再定義（top_pの処理で使うため）
            # top_k > 0 の場合、top_kで絞った後の値を使う。0の場合は全範囲。
            top_k_logits, top_k_indices = logits, torch.arange(logits.size(-1), device=logits.device).expand(logits.size(0), -1)
            # ↑ このアプローチはメモリを食うため、元の実装の通りtop_k_indicesでmaskする方が綺麗です。
            # 以下、元のロジックを活かした修正版です。
            # 2. Top-P (Nucleus) filtering
            if self.top_p < 1.0:
                # Top-K/Allで絞ったテンソルでソート
                sorted_logits, sorted_indices = torch.sort(top_k_logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                # 除去対象のマスクを作成
                sorted_indices_to_remove = cumulative_probs > self.top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                # top_k_logitsと同じ形状のマスクを作成
                indices_to_remove_k = torch.zeros_like(top_k_logits, dtype=torch.bool).scatter_(
                    dim=-1, index=sorted_indices, src=sorted_indices_to_remove
                )
                top_k_logits[indices_to_remove_k] = -float('Inf')

            # 3. 全語彙の logits を一旦すべて -inf にして、生き残った top_k_logits だけを戻す
            new_logits = torch.full_like(logits, float('-inf'))
            new_logits.scatter_(dim=-1, index=top_k_indices, src=top_k_logits)
            logits = new_logits

            if torch.all(logits == float('inf')):
                logits = torch.ones_like(logits)  # テンソル全体を1にする場合

            probs = F.softmax(logits, dim=-1) # B*W*N,C この softmax は、C についての softmax

            #torch.manual_seed(42)
            _index_flat = torch.multinomial(probs, num_samples=self.cand, replacement=False) #B*N*W,cand
            #_index_flat = torch.topk( probs, self.cand, dim = -1 )
            _index_flat1 = _index_flat[:,:1]

            #_score_flat = torch.gather( probs, -1, _index_flat1)
            #_score_flat2 = torch.gather( probs, -1, _index_flat)
            #_score_flat = torch.gather( flat_score / self.temp, -1, _index_flat1)
            #_score_flat2 = torch.gather( flat_score / self.temp, -1, _index_flat)
            _score_flat = torch.gather( flat_score, -1, _index_flat1)
            #_score_flat2 = torch.gather( flat_score, -1, _index_flat)
            _index2 = _index_flat.view(B, N, W, self.cand) # B, N, W, cand
            _score = _score_flat.view(B, N, W) # B, N, W
            #_score2 = _score_flat2.view( B, N, W, self.cand )

            #_score = _score + beam_emission_scores[:, i][:,None,:].expand(-1,N,-1)

            if masks is not None:
                #print( "masks[:,:,i:i+1].size():",masks[:,:,i:i+1].size())
                #print( "_score.size():", _score.size() )
                #print( "score.size():", score.size() )
                score = torch.where(masks[:,:, i: i+1], _score, score)
                #index = torch.where(masks[:, i: i+1], _index, dummy)
                #print( "_index2.size():", _index2.size() )
                index2 = torch.where(masks[:,:, i: i+1][:,:,:,None].expand( -1, -1, -1, self.cand ), _index2, dummy2 )
            else:
                score, index2 = _score, _index2


            #score = _score
            #_score2 = _score2 + beam_emission_scores[:,i][:,None,:,None].expand( -1, N, -1, self.cand )
            #score2 = _score2

            traj_tokens.append( index2 ) # S, B, N, W, cand

        ## 2. サンプリングされた最後のインデックスを取得 (B, N, 1)
        #print( "_score:", _score )
        B, N, C = score.shape
        flat_score = score.reshape(-1, C)

        probs = F.softmax(flat_score / self.temp, dim=-1)
        #torch.manual_seed(42)
        #print( "probs:", probs )
        _index_flat = torch.multinomial(probs, num_samples=self.cand, replacement = False ) #B*N,cand
        #_index_flat = torch.topk( probs, self.cand, -1 )

        #_score_flat = torch.gather(probs, -1, _index_flat)
        #_score_flat = torch.gather(flat_score, -1, _index_flat)
        _index = _index_flat.view(B, N, self.cand)
        #_score = _score_flat.view(B, N, self.cand)
        #_score = _score.unsqueeze(2).expand(-1,-1,W,-1)
        current_sampled_index = _index #(B, N, cand )

        # beam から vocab_size に戻す。
        N_beam_targets = beam_targets[:,-1].unsqueeze( 1 ).expand( -1, N, -1 ) # B, N, W
        current_sampled_index = torch.gather( N_beam_targets, -1, current_sampled_index ) #B,N,cand

        finalized_tokens = torch.full( (seq_len,B,N), self.vocab_size , dtype=torch.long, device=device)

        ## 3. 最初の要素として追加
        finalized_tokens[0] = current_sampled_index[:,:,0]
        #traj_scores = torch.stack( traj_scores, dim = 0 )
        traj_tokens = torch.stack( traj_tokens, dim = 0 )

        # beam から vocab_size に戻す。
        N_beam_targets = beam_targets.unsqueeze( 2 ).unsqueeze(4).expand( -1, -1, N, -1, self.cand ) # B, seq_len, N, W
        N_beam_targets = N_beam_targets.permute( 1, 0, 2, 4, 3 ) #S,B,N,cand,W
        N_beam_targets1 = N_beam_targets[:-1]
        N_beam_targets2 = N_beam_targets[1:]
        ##traj_tokens S,B,N,W,cand, traj_scores S,B,N,W
        traj_tokens = torch.gather( N_beam_targets1, -1, traj_tokens.transpose(3,4))
        traj_tokens3 = torch.full( ( seq_len - 1, B, N, self.cand, self.vocab_size ), self.vocab_size, dtype=torch.long, device = beam_targets.device )
        traj_tokens3 = torch.scatter( traj_tokens3, -1, index = N_beam_targets2, src = traj_tokens)

        # バックトレーシング
        if sampled_beam_idx_flag:
            beam_logits = []
        #for i3, (idx_step, prob_step ) in enumerate( zip(torch.flip(traj_tokens3, dims=(0,)),torch.flip(traj_scores, dims=(0,)) ) ):
        for i3, (idx_step, step_prob ) in enumerate( zip(torch.flip(traj_tokens3, dims=(0,)), reversed( step_probs ))):
            if sampled_beam_idx_flag:
                i4 = seq_len - i3 - 1
                previous_beam_index = sampled_beam_idx[:,i4-1]
                prob_from_prev = step_prob.gather(2, previous_beam_index[:, :, None, None].expand(-1, -1, 1,  beam))
                beam_logits.append( prob_from_prev.squeeze(2) ) # (S), B, W

            i2= i3+1
            previous_pointer = finalized_tokens[i3]
            stop_flag = torch.zeros( (B,N), dtype=torch.int,device=device)
            for i in range( self.cand ):
                #cand_probs = prob_step[:,:,:,i]
                idx = idx_step[:,:,i].new_empty((idx_step[:,:,i].size(0), idx_step[:,:,i].size(1), idx_step[:,:,i].size(2)+1))
                idx[:,:,:-1] = idx_step[:,:,i]
                idx[:,:,idx.size(2)-1] = self.vocab_size
                cand_tokens = torch.gather( idx,-1, previous_pointer.unsqueeze(-1) ).squeeze(-1)# B,N
                finalized_tokens2 = finalized_tokens.clone()
                finalized_tokens2[ i2: ] = -100
                repeat_mask = ( finalized_tokens2 == cand_tokens  )  # S, B , N
                not_permit_mask = (~torch.isin( finalized_tokens2, torch.tensor( permit_repeat, device=device))).to(torch.int )
                repeat_sum = ( (repeat_mask).to(torch.int) * not_permit_mask ).sum(dim =0 )# B,N
                if i == self.cand -1:#最終の時は、
                    chg_flag = ~(stop_flag.to(torch.bool)) #最終の前までに stop が1になれば変えない。stop が0だったら変える。
                    if i == 0:
                        chg_flag = torch.ones( (B,N), dtype=torch.bool,device=device)
                else:
                    tmp_flag =  stop_flag + repeat_sum  # stopとrepeat両方が0の時 0
                    chg_flag = ( tmp_flag == 0 )# stop と　repeat両方が0の時　chgは　true
                    stop_flag[ stop_flag == 1 ] = 1
                    stop_flag[ chg_flag ] = 1
                finalized_tokens[i2] = torch.where(chg_flag,cand_tokens,finalized_tokens[i2])#(S),B,N True だったら変更、 False だったらそのまま。
                chg_flag2 = chg_flag.unsqueeze(2).expand(-1,-1,W)

        finalized_tokens = torch.flip( finalized_tokens, dims= (0, ) )
        finalized_tokens = finalized_tokens.transpose( 0, 1 )#B,S,N

        if not sampled_beam_idx_flag:

            # vocab_size のfinalized_tokens から beam の sampled_beam_idx を作る
            N_beam_targets = beam_targets.unsqueeze( 2 ).expand( -1, -1, N, -1 ) # B, seq_len, N, W
            mask = N_beam_targets == finalized_tokens.unsqueeze(-1) #B,S,N,W
            # 2. ビーム次元 (-1) で一致しているインデックスを取得
            sampled_beam_idx = torch.argmax(mask.to(torch.int32), dim=-1)

            beam_logits = []
            for i3, step_prob in enumerate( reversed( step_probs )):
                i4 = seq_len - i3 - 1
                previous_beam_index = sampled_beam_idx[:,i4-1]
                prob_from_prev = step_prob.gather(2, previous_beam_index[:, :, None, None].expand(-1, -1, 1,  beam))
                beam_logits.append( prob_from_prev.squeeze(2) ) # (S), B, W

        #print( "log_beam_logits:", log_beam_logits )
        beam_logits.append( logits_t0 )
        beam_logits = torch.stack( beam_logits, dim = 0 )
        beam_logits = torch.flip( beam_logits, dims = (0, ) )
        beam_logits = beam_logits.permute( 1, 0, 3, 2 ) # B,S,W,N
        beam_probs = F.softmax( beam_logits, dim = 2 )

        log_beam_probs = torch.log( beam_probs + eps )


        return beam_probs, log_beam_probs, sampled_beam_idx, finalized_tokens

    ''' original
    def _viterbi_decode(self, emissions, masks=None, beam=None):
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam)

        traj_tokens, traj_scores = [], []
        finalized_tokens, finalized_scores = [], []

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

        for i in range(1, seq_len):
            traj_scores.append(score)
            _score = score[:, :, None] + beam_transition_matrix[:, i-1]
            _score, _index = _score.max(dim=1)
            _score = _score + beam_emission_scores[:, i]

            if masks is not None:
                score = torch.where(masks[:, i: i+1], _score, score)
                index = torch.where(masks[:, i: i+1], _index, dummy)
            else:
                score, index = _score, _index
            traj_tokens.append(index)

        # now running the back-tracing and find the best
        best_score, best_index = score.max(dim=1)
        finalized_tokens.append(best_index[:, None])
        finalized_scores.append(best_score[:, None])

        for idx, scs in zip(reversed(traj_tokens), reversed(traj_scores)):
            previous_index = finalized_tokens[-1]
            finalized_tokens.append(idx.gather(1, previous_index))
            finalized_scores.append(scs.gather(1, previous_index))

        finalized_tokens.reverse()
        finalized_tokens = torch.cat(finalized_tokens, 1)
        finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

        finalized_scores.reverse()
        finalized_scores = torch.cat(finalized_scores, 1)
        finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]

        return finalized_scores, finalized_tokens
    '''
    def forward(self, emissions, targets, sampled_beam_idx = None, top_indices = None, masks=None, grpo_mode = True, crf_mode = False, 
                use_crf_beam_logits = False,  beam=None, repeat_penalty = 0.04  ):

        #print( "emissions:", emissions )
        
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        B = batch_size
        device = emissions.device

        permit_repeat = [ pad_token_id, eos_token_id, cls_token_id, sep_token_id, a_token_id, an_token_id, the_token_id, period_token_id, \
                         comma_token_id, and_token_id, in_token_id, we_token_id, i_token_id, he_token_id, she_token_id, \
                         it_token_id, they_token_id, dbl_token_id, sgl_token_id ]
        permit_repeat = torch.tensor( permit_repeat ).to(device)
        
        if top_indices == None:
            beam_emission_scores, beam_targets = torch.topk( emissions, beam, -1)
        else:
            beam_emission_scores = torch.gather( emissions, -1, top_indices )
            beam_targets = top_indices
        
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, self.beam, self.rank),
            beam_transition_score2.view(-1, self.beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam
        
        if self.cand == 0:
            traj_tokens = []
            step_probs = []
            finalized_tokens = []
            
            # compute the normalizer in the log-space
            score = beam_emission_scores[:, 0]  # B x K
            dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()
            logits_t0 = score / self.temp
            
            beam_history = beam_targets[:, 0].unsqueeze(-1) # [B, beam, 1]
    
            for i in range(1, seq_len):
                # _score_matrix 形状: [B, beam_from, beam_to]
                _score_matrix = score.unsqueeze(-1) + beam_transition_matrix[:, i-1, :, :]
                _score_matrix = _score_matrix + beam_emission_scores[:, i].unsqueeze(1)
    
                if repeat_penalty > 0.0:
                    # to_tokens shape: [B, 1, beam_to]
                    to_tokens = beam_targets[:, i].unsqueeze(1)
    
                    # is_permitted shape: [B, 1, beam_to]
                    is_permitted = torch.isin(to_tokens, permit_repeat)
    
                    # history shape: [B, beam_from, past length, 1]
                    # target  shape: [B, 1,         1,         beam_to]
                    history_expanded = beam_history.unsqueeze(-1)
                    to_tokens_expanded = to_tokens.unsqueeze(2)
    
                    # match_matrix shape: [B, beam_from, past length, beam_to]
                    match_matrix = (history_expanded == to_tokens_expanded)
    
                    # repeat_mask shape: [B, beam_from, beam_to]
                    repeat_mask = match_matrix.any(dim=2) & (~is_permitted)
                    #repeat_count = torch.sum( match_matrix.float(), dim = 2 )
                    #repeat_count = repeat_count * is_permitted.float()

                    #log_penalty = torch.log( torch.tensor( repeat_penalty))
                    #penalty_matrix = log_penalty * repeat_count
                    #penalty_delta = torch.where( repeat_count != 0.0, penalty_matrix, 0.0 )
                    penalty_matrix = repeat_mask.float()
                    log_penalty = torch.log( torch.tensor( repeat_penalty))
                    penalty_delta = torch.where( penalty_matrix == 1.0, log_penalty, 0.0)
                    _search_score_matrix = _score_matrix + penalty_delta
                else:
                    _search_score_matrix = _score_matrix
    
                step_probs.append(_search_score_matrix / self.temp)
    
                # _score: [B, beam_to], _index: [B, beam_to] 
                _score, _index = _search_score_matrix.max(dim=1)
    
                if masks is not None:
                    score = torch.where(masks[:, i: i+1], _score, score)
                    index = torch.where(masks[:, i: i+1], _index, dummy)
                else:
                    score, index = _score, _index
    
                traj_tokens.append(index)
    
    
                history_len = beam_history.size(-1)
                gather_index = index.unsqueeze(-1).expand(-1, -1, history_len)
                updated_history = torch.gather(beam_history, 1, gather_index)
    
                current_tokens = beam_targets[:, i].unsqueeze(-1)
                beam_history = torch.cat([updated_history, current_tokens], dim=-1)
    
            _, _index = torch.max(score, dim=1)
            current_sampled_index = _index
            finalized_tokens.append(current_sampled_index.unsqueeze(-1))
    
            for idx in reversed(traj_tokens):
                previous_index = finalized_tokens[-1]
                finalized_tokens.append(idx.gather(1, previous_index))
    
            finalized_tokens.reverse()
            finalized_tokens = torch.cat(finalized_tokens, 1)
            finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

        else:    
            traj_tokens = []
            step_probs = []

            # compute the normalizer in the log-space
            score = beam_emission_scores[:, 0]  # B x K
            dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()
            dummy2 = dummy[:,None,:].expand( -1, self.cand, -1 )
            
            logits_t0 = score  / self.temp            
            
            for i in range(1, seq_len):
                _score_matrix = score.unsqueeze(-1) + beam_transition_matrix[:,i-1,:,:,].expand( -1, -1, -1 )
                _score_matrix = _score_matrix + beam_emission_scores[:,i][:,None,:].expand(-1,beam,-1)

                step_probs.append( _score_matrix / self.temp )

                _score2, _index2 = torch.topk( _score_matrix, self.cand, dim = 1 ) 
                _score = _score2[:,0]
                _index = _index2[:,0]
                
                if masks is not None:
                    score = torch.where(masks[:, i: i+1], _score, score)
                    index = torch.where(masks[:, i: i+1], _index, dummy)
                    index2 = torch.where(masks[:, i: i+1][:,None].expand( -1, self.cand, -1 ), _index2, dummy2 )
                else:
                    score, index, index2 = _score, _index, _index2
                
                traj_tokens.append(index2) # S, B, cand, W

            _, _indexes = torch.topk( score, self.cand, dim = 1 )
            current_sampled_index = _indexes #(B,cand )

            # beam から vocab_size に戻す。
            beam_targets1 = beam_targets[:,-1] # B, W
            current_sampled_index = torch.gather( beam_targets1, -1, current_sampled_index ) #B,cand
        
            finalized_tokens = torch.full( (seq_len,B), self.vocab_size , dtype=torch.long, device=device)
    
            ## 3. 最初の要素として追加
            finalized_tokens[0] = current_sampled_index[:,0] # (S), B

            traj_tokens = torch.stack( traj_tokens, dim = 0 ) # S, B, cand, W
            
            # beam から vocab_size に戻す。
            cand_beam_targets = beam_targets.unsqueeze(2).expand( -1, -1, self.cand, -1 ) # B, seq_len, cand, W
            cand_beam_targets = cand_beam_targets.permute( 1, 0, 2, 3 ) #S,B,cand,W
            cand_beam_targets1 = cand_beam_targets[:-1]
            cand_beam_targets2 = cand_beam_targets[1:]
            traj_tokens = torch.gather( cand_beam_targets1, -1, traj_tokens)
            traj_tokens3 = torch.full( ( seq_len - 1, B, self.cand, self.vocab_size ), self.vocab_size, dtype=torch.long, device = beam_targets.device )
            traj_tokens3 = torch.scatter( traj_tokens3, -1, index = cand_beam_targets2, src = traj_tokens)

            for i3, idx_step in enumerate( torch.flip(traj_tokens3, dims=(0,))):
                i2= i3+1
                previous_pointer = finalized_tokens[i3]
                stop_flag = torch.zeros( (B), dtype=torch.int,device=device) # stop_flag が 0 の時 更新 OK, 1の時、これ以上更新しない。
                for i in range( self.cand ):
                    idx = idx_step[:,i].new_empty((idx_step[:,i].size(0), idx_step[:,i].size(1)+1 ))
                    idx[:,:-1] = idx_step[:,i]
                    idx[:,idx.size(1)-1] = self.vocab_size
                    cand_tokens = torch.gather( idx,-1, previous_pointer.unsqueeze(-1) ).squeeze(-1)# B,N　更新の候補を作成。
                    finalized_tokens2 = finalized_tokens.clone()
                    finalized_tokens2[ i2: ] = -100 # 現在の時刻より先は -100
                    repeat_mask = ( finalized_tokens2 == cand_tokens  )  # S, B 繰り返しの場所を特定する　mask 
                    not_permit_mask = (~torch.isin( finalized_tokens2, torch.tensor( permit_repeat, device=device))).to(torch.int ) 
                    repeat_sum = ( (repeat_mask).to(torch.int) * not_permit_mask ).sum(dim =0 )# B 繰り返しの場所に繰り返しが許可されたトークンの場所をかけて和をとることにより、B の形状の繰り返しがある時1 以上、ない時0 を 得る。
                    if i == self.cand -1:#最終の時は、
                        chg_flag = ~(stop_flag.to(torch.bool)) #最終の前までに stop が1になれば変えない。stop が0だったら変える。
                        if i == 0:
                            chg_flag = torch.ones( (B), dtype=torch.bool,device=device)
                    else:
                        tmp_flag =  stop_flag + repeat_sum  # stopとrepeat両方が0の時 0
                        chg_flag = ( tmp_flag == 0 )# stop と　repeat両方が0の時　chgは　true
                        stop_flag[ stop_flag == 1 ] = 1 # stop_flag が 1の場合は stop_flag =1
                        stop_flag[ chg_flag ] = 1 #chg_flag = True の場合は、更新されたのだから stop_flag = 1
                    finalized_tokens[i2] = torch.where(chg_flag,cand_tokens,finalized_tokens[i2])#(S),B True だったら変更、 False だったらそのまま。
                    chg_flag2 = chg_flag.unsqueeze(1).expand(-1,beam)
            
            finalized_tokens = torch.flip( finalized_tokens, dims= (0, ) )
            finalized_tokens = finalized_tokens.transpose( 0, 1 )#B,S,N

        if use_crf_beam_logits:
            # vocab_size のfinalized_tokens から beam の sampled_beam_idx を作る
            mask = beam_targets == finalized_tokens.unsqueeze(-1) #B,S,W
            # 2. ビーム次元 (-1) で一致しているインデックスを取得
            sampled_beam_idx2 = torch.argmax(mask.to(torch.int32), dim=-1)
    
            beam_logits = []
            #print( "define log_beam_probs None:")
            for i3, probs_step in enumerate( reversed( step_probs )):
                i4 = seq_len - i3 - 1
                #current_beam_index = sampled_beam_idx2[:,i4]
                previous_beam_index = sampled_beam_idx2[:,i4-1]
                prob_from_prev = probs_step.gather(2, previous_beam_index[:, None, None].expand(-1, 1,  beam))
                beam_logits.append( prob_from_prev.squeeze(1) )
    
            beam_logits.append( logits_t0 )
            beam_logits = torch.stack( beam_logits, dim = 0 )
            beam_logits = torch.flip( beam_logits, dims = (0, ) )
            beam_logits = beam_logits.permute( 1, 0, 2 )
            crf_beam_logits = beam_logits            
        else:
            crf_beam_logits = torch.tensor( [0] )

        if not self.sft_mode:
            if grpo_mode:
                if not use_crf_beam_logits:
                    numerator = self._compute_score(emissions, targets, masks)
                    denominator = self._compute_normalizer(emissions, targets, masks, beam)
                    crf_loss = - ( ( numerator - denominator ) / torch.sum( ( masks == True ).float(), dim = 1 ) ) .mean( dim = 0 )
                    crf_beam_logits = torch.tensor([0])
                else:
                    crf_loss = torch.tensor([0])
                if self.cand == 0:
                    beam_probs, log_beam_probs, sampled_beam_idx, b_finalized_tokens, = \
                        self._compute_grpo_samples(beam_emission_scores, beam_transition_matrix, beam_targets, sampled_beam_idx=sampled_beam_idx, repeat_penalty=repeat_penalty)
                else:               
                    beam_probs, log_beam_probs, sampled_beam_idx, b_finalized_tokens, = \
                        self._compute_grpo_samples_suppression(beam_emission_scores, beam_transition_matrix, beam_targets, sampled_beam_idx=sampled_beam_idx)
                return finalized_tokens, beam_probs, log_beam_probs, crf_loss, crf_beam_logits, beam_targets, sampled_beam_idx, b_finalized_tokens
            else:
                if crf_mode:
                    if not use_crf_beam_logits:
                        numerator = self._compute_score(emissions, targets, masks)
                        denominator = self._compute_normalizer(emissions, targets, masks, beam)
                        crf_loss = - ( ( numerator - denominator ) / torch.sum( ( masks == True ).float(), dim = 1 ) ) .mean( dim = 0 )
                        crf_beam_logits = torch.tensor([0])
                    else:
                        crf_loss = torch.tensor([0])
                    return finalized_tokens, crf_loss, crf_beam_logits, beam_targets
                else:
                    return finalized_tokens
        else:
            if not use_crf_beam_logits:
                numerator = self._compute_score(emissions, targets, masks)
                denominator = self._compute_normalizer(emissions, targets, masks, beam)
                crf_loss = - ( ( numerator - denominator ) / torch.sum( ( masks == True ).float(), dim = 1 ) ) .mean( dim = 0 )
                crf_beam_logits = torch.tensor([0])
            else:
                crf_loss = torch.tensor([0])
            return finalized_tokens, crf_loss, crf_beam_logits , beam_targets

    '''
    #original　学習の確認用に使った
    #def _viterbi_decode(self, emissions, masks=None, beam=None):
    def forward(self, emissions, targets, sampled_beam_idx = None, top_indices = None, masks=None, grpo_mode = True, crf_mode = False,
                use_crf_beam_logits = False,  beam=None, repeat_penalty = 0.04  ):
        
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        traj_tokens, traj_scores = [], []
        finalized_tokens, finalized_scores = [], []

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        #print( "score size:", score.size() )
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

        for i in range(1, seq_len):
            traj_scores.append(score)
            _score = score[:, :, None] + beam_transition_matrix[:, i-1] # bsz, beam, beam
            _score, _index = _score.max(dim=1) # bsz, beam     bsz, beam
            _score = _score + beam_emission_scores[:, i] # bsz, beam

            if masks is not None:
                score = torch.where(masks[:, i: i+1], _score, score)
                index = torch.where(masks[:, i: i+1], _index, dummy)
            else:
                score, index = _score, _index
            traj_tokens.append(index)

        # now running the back-tracing and find the best
        best_score, best_index = score.max(dim=1)
        finalized_tokens.append(best_index[:, None])
        finalized_scores.append(best_score[:, None])

        for idx, scs in zip(reversed(traj_tokens), reversed(traj_scores)):
            previous_index = finalized_tokens[-1]
            finalized_tokens.append(idx.gather(1, previous_index))
            finalized_scores.append(scs.gather(1, previous_index))

        finalized_tokens.reverse()
        finalized_tokens = torch.cat(finalized_tokens, 1)
        finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

        finalized_scores.reverse()
        finalized_scores = torch.cat(finalized_scores, 1)
        finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]

        #numerator = self._compute_score(emissions, targets)
        #denominator = self._compute_normalizer(emissions, targets)
        #print( "emissions.size():", emissions.size() )
        #print( "masks:size():", masks.size() )
        numerator = self._compute_score(emissions, targets, masks)
        #print( " seq_len:", seq_len )
        #print( " torch.sum( ( masks == True ).float(), dim = 1 ):", torch.sum( ( masks == True ).float(), dim = 1 ) )
        #numerator = numerator / torch.sum( ( masks == True ).float(), dim = 1 )
        denominator = self._compute_normalizer(emissions, targets, masks, beam)

        crf_loss = - ( ( numerator - denominator ) / torch.sum( ( masks == True ).float(), dim = 1 ) ) .mean( dim = 0 )

        crf_beam_logits = torch.tensor( [0] )
        
        return finalized_tokens, crf_loss, crf_beam_logits , beam_targets
        
        #return finalized_scores, finalized_tokens
    '''
    ''' 強化学習用の log_probs 計算あり。
    def _viterbi_decode(self, emissions, masks=None, beam=None):
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        traj_tokens, traj_scores = [], []
        finalized_tokens, finalized_scores = [], []

        # 時刻ごとの遷移確率行列（log_softmax）を保存するリスト
        # 各要素の形状: (bsz, beam, beam) -> [batch, from_beam_idx, to_beam_idx]
        step_log_probs = []

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

        # 時刻 t=0 の初期選択確率（最高位候補の中でのlog_softmax）
        # 形状: (bsz, beam)
        log_probs_t0 = torch.log_softmax(score / self.temp, dim=-1)

        for i in range(1, seq_len):
            traj_scores.append(score)

            # _score_matrix 形状: (bsz, beam, beam) -> [batch, from_beam, to_beam]
            _score_matrix = score[:, :, None] + beam_transition_matrix[:, i-1]
            _score_matrix = _score_matrix + beam_emission_scores[:, i][:, None, :]

            # 【重要】強化学習用の遷移対数確率（次のBeam候補 dim=-1 に対する確率分布）
            # 形状: (bsz, beam, beam)
            action_log_prob = torch.log_softmax(_score_matrix / self.temp, dim=-1)
            #action_log_prob = _score_matrix
            #action_log_prob = _score_matrxi / self.temp
            step_log_probs.append(action_log_prob)

            # 通常のビタビ処理
            _score, _index = _score_matrix.max(dim=1) # bsz, beam

            if masks is not None:
                score = torch.where(masks[:, i: i+1], _score, score)
                index = torch.where(masks[:, i: i+1], _index, dummy)
            else:
                score, index = _score, _index
            traj_tokens.append(index)

        #traj_scores.append(score)

        # --- バックトラック（逆順のトレース） ---
        best_score, best_index = score.max(dim=1)
        finalized_tokens.append(best_index[:, None])
        finalized_scores.append(best_score[:, None])

        # 強化学習用の確率を後ろから格納していくリスト
        sampled_log_probs_reversed = []

        # zipの中身: i が大きい（未来）順にループが回る
        # idx: 時刻 i で選択された「前の時刻のBeamインデックス」 (bsz, beam)
        # scs: 時刻 i-1 での累積スコア (bsz, beam)
        # step_probs: 時刻 i-1 から i への遷移対数確率 (bsz, beam, beam)
        for idx, scs, step_probs in zip(reversed(traj_tokens), reversed(traj_scores), reversed(step_log_probs)):
            # 現在（時刻 i）確定している最適パスの「現在のBeamインデックス」 (bsz, 1)
            current_beam_index = finalized_tokens[-1]

            # 1. 1つ前の時刻の最適Beamインデックスを取得
            previous_beam_index = idx.gather(1, current_beam_index)
            finalized_tokens.append(previous_beam_index)
            finalized_scores.append(scs.gather(1, current_beam_index))

            # 2. 【重要】強化学習用確率の抽出
            # step_probs は (bsz, from_beam, to_beam)
            # a) まず from_beam (dim=1) を previous_beam_index で指定して抽出 -> (bsz, 1, beam)
            prob_from_prev = step_probs.gather(1, previous_beam_index[:, :, None].expand(-1, -1, beam))
            # b) 次に to_beam (dim=-1) を current_beam_index で指定して抽出 -> (bsz, 1, 1)
            prob_sampled = prob_from_prev.gather(2, current_beam_index[:, :, None])

            sampled_log_probs_reversed.append(prob_sampled.squeeze(2)) # (bsz, 1)

        # 時刻 t=0 の確率を抽出して追加
        # finalized_tokens[-1] には、最初の時刻 t=0 で選ばれた最適パスのBeamインデックスが入っている
        t0_index = finalized_tokens[-1]
        prob_t0 = log_probs_t0.gather(1, t0_index) # (bsz, 1)
        sampled_log_probs_reversed.append(prob_t0)

        # 通常のトークンIDの復元処理
        finalized_tokens.reverse()
        finalized_tokens = torch.cat(finalized_tokens, 1)
        finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

        finalized_scores.reverse()
        finalized_scores = torch.cat(finalized_scores, 1)
        finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]

        # 強化学習用の確率を時間順に戻して結合
        sampled_log_probs_reversed.reverse()
        # 形状: (bsz, seq_len, 1)
        sampled_log_probs = torch.cat(sampled_log_probs_reversed, dim=1)[:, :, None]

        return finalized_scores, finalized_tokens, sampled_log_probs
    '''



In [ ]:
class PositionalEmbedding(nn.Module):
    '''
    位置埋め込み （Positional embedding）
    dim_embedding: 埋込み次元
    max_len      : 入力の最大系列長
    '''
    def __init__(self, dim_embedding: int, max_len: int=2048):
        super().__init__()

        self.pos_emb = nn.Embedding(max_len, dim_embedding)

    '''
    位置エンコーディングの順伝播
    x: 位置エンコーディングを埋め込む対象のテンソル,
       [バッチサイズ, 系列長, 埋め込み次元]
    '''
    def forward(self, x: torch.Tensor):
        seq = x.shape[1]
        positions = torch.arange(start=0, end=seq, step=1, device=x.device).to(torch.long)
        positions = self.pos_emb(positions)[:seq,:]
        
        return positions

In [ ]:
class BertCRF(nn.Module):
    
    def __init__(self, tokenizer, device = torch.device( 'cpu' ), max_target_length = 128, crf_coef = 1.0, temp = 1.0, \
                 top_p = 0.9, top_k = 50, num_samples = 4, cand = 4, sft_mode = False  ):
        super().__init__()

        self.sft_mode = sft_mode
        
        dim_embedding = 1024
        self.dim_embedding = dim_embedding
        nhead = 16
        d_layers = 24
        self.cand = cand
        
        self.tgt_embedding = nn.Embedding( len(tokenizer), dim_embedding )
        self.tgt_pos_emb = PositionalEmbedding( dim_embedding )
        
        model_checkpoint = "google-bert/bert-large-uncased"
        self.bert = BertModel.from_pretrained( model_checkpoint ).to(device)
        self.max_target_length = max_target_length
        decoder_layer = nn.TransformerDecoderLayer( d_model = dim_embedding, nhead = nhead, batch_first=True  ).to(device)
        self.transformer_decoder = nn.TransformerDecoder( decoder_layer, num_layers = d_layers ).to(device)
        
        vocab_size = len( tokenizer )
        self.emission_outputs = nn.LayerNorm( dim_embedding ).to(device)
        self.emission_linear = nn.Linear(dim_embedding, vocab_size).to(device)

        self.length_prediction_linear = nn.Linear( dim_embedding, max_target_length + 1  ).to(device)

        
        crf_low_rank = 32
        crf_beam = 256
        
        self.crf_layer = DynamicCRF(num_embedding = vocab_size, low_rank = crf_low_rank, beam_size = crf_beam, 
                            crf_coef=crf_coef, temp=temp, top_p = top_p, top_k = top_k, num_samples= num_samples, cand = cand,
                            sft_mode = sft_mode ).to(device)

    def forward( self, input_ids, attention_mask, labels, sampled_beam_idx = None, top_indices = None, grpo_mode = False, 
                 crf_mode = True, use_crf_beam_logits = False, p_glat = 0.0, is_inference = False, 
                 use_tgt_interpolate = 'none', repeat_penalty = 0.04  ):

        memory = self.bert( input_ids = input_ids, attention_mask = attention_mask ).last_hidden_state

        predicted_text_logits = self.length_prediction_linear( memory[:,0,:] )

        predicted_length = torch.argmax( predicted_text_logits, dim = -1 )
        #teacher_length = torch.sum( torch.ne( labels, tokenizer.pad_token_id ).int(), dim = -1  )

        #text_length = torch.zeros_like( teacher_length )
        
        ## teacher_length と同じ形状のランダムな値（0〜1）のテンソルを作成
        #mask = torch.rand_like(teacher_length, dtype=torch.float) < p_pred
        
        ## 条件がTrueならpredicted_length、Falseならteacher_lengthを選択
        #text_length = torch.where(mask, predicted_length, teacher_length)        

        if is_inference:
            text_length = torch.argmax( predicted_text_logits, dim = -1 )
        else:
            text_length = torch.sum( torch.ne( labels, tokenizer.pad_token_id ).to(torch.int), dim = -1 )
        
        # --- 推論時：予測された長さに基づいて tgt とマスクを作成 ---
        arange_seq = torch.arange(self.max_target_length, device=input_ids.device).unsqueeze(0)
        mask_condition = arange_seq < text_length.unsqueeze(1)
        
        tgt = torch.full( ( memory.size(0), self.max_target_length ), pad_token_id,  device = input_ids.device, dtype = torch.int )
        tgt = torch.where(mask_condition, mask_token_id, tgt)
        
        # 推論時の特殊トークン付与（必要に応じて）
        tgt[:,0] = cls_token_id
        for i, length in enumerate( text_length ):
            # labelsの最大長を超えないよう安全弁を入れる
            safe_length = min(length.item(), self.max_target_length)
            if safe_length >= 3:
                tgt[i, safe_length-1] = eos_token_id
                tgt[i, safe_length-2] = eos_token_id
                tgt[i, safe_length-3] = sep_token_id
                
        tgt_key_padding_mask = tgt.eq( tokenizer.pad_token_id ).bool()

        if use_tgt_interpolate == 'none':
            if is_inference:
                embed_tgt = self.tgt_embedding( tgt )
            else:
                if p_glat > 0.0 and p_glat <= 0.8 :
                    arange_seq = torch.arange(self.max_target_length, device=input_ids.device).unsqueeze(0)
                    mask_condition = arange_seq < text_length.unsqueeze(1)
                    # 1である場所に対して確率 p で 1、(1-p) で 0 を生成するマスクを作る
                    # torch.rand_like または torch.bernoulli を使用
                    random_mask = (torch.rand_like(mask_condition.float()) < p_glat).float()
                    
                    # 元のテンソルが 1 の部分だけに適用
                    result = mask_condition * random_mask
        
                    tgt = torch.where( result.bool(), labels, tgt )
        
                    embed_tgt = self.tgt_embedding( tgt )
                else:
                    embed_tgt = self.tgt_embedding( tgt )
        elif use_tgt_interpolate == 'input_ids':
            embed_tgt = self.tgt_embedding( tgt )
            input_length = torch.sum( torch.ne( input_ids, tokenizer.pad_token_id ).to(torch.int), dim = -1  ) 
            tgt_length = torch.sum( torch.ne( tgt, tokenizer.pad_token_id ).to(torch.int), dim = -1 )
            for i2, (ids, length_input, length_tgt) in enumerate( zip( input_ids, input_length, tgt_length ) ):
                if length_tgt > 4 and length_input > 4:
                    each_input_ids2 = ids[1:length_input-3]
                    each_memory2 = self.tgt_embedding( each_input_ids2[None] )
                    tmp_vector = F.interpolate( each_memory2.permute( 0,2,1 ), size=length_tgt - 4, mode='linear', align_corners=True ).permute( 0, 2, 1 ).squeeze(0)
                    embed_tgt[i2, 1:length_tgt - 3, :] = tmp_vector
        else:
            embed_tgt = self.tgt_embedding( tgt )
            input_length = torch.sum( torch.ne( input_ids, tokenizer.pad_token_id ).to(torch.int), dim = -1  ) 
            tgt_length = torch.sum( torch.ne( tgt, tokenizer.pad_token_id ).to(torch.int), dim = -1 )
            for i2, (mem, length_input, length_tgt) in enumerate( zip( memory, input_length, tgt_length ) ):
                if length_tgt > 4 and length_input > 4:
                    each_memory2 = mem[1:length_input - 3, : ][None] 
                    tmp_vector = F.interpolate( each_memory2.permute( 0,2,1 ), size=length_tgt - 4, mode='linear', align_corners=True ).permute( 0, 2, 1 ).squeeze(0)
                    embed_tgt[i2, 1:length_tgt - 3, :] = tmp_vector
        
        pos_emb_tgt = embed_tgt + self.tgt_pos_emb( embed_tgt )
        
        outputs = self.transformer_decoder( pos_emb_tgt, memory, memory_key_padding_mask = ~(attention_mask.to(torch.bool)),
                                          tgt_key_padding_mask = tgt_key_padding_mask )
        
        outputs = self.emission_outputs( outputs )
        emissions = self.emission_linear( outputs )

        #emission_mask = ~input_ids.eq(tokenizer.pad_token_id)
        emission_mask = ~tgt.eq( tokenizer.pad_token_id ).bool()

        if not self.sft_mode:
            if not grpo_mode and crf_mode:
                finalized_tokens, crf_loss, crf_beam_logits, beam_targets = self.crf_layer( emissions, labels, masks = emission_mask,
                                                                                        grpo_mode = grpo_mode, crf_mode = crf_mode,
                                                                                        use_crf_beam_logits = use_crf_beam_logits,
                                                                                        repeat_penalty = repeat_penalty )
                return finalized_tokens, emissions, crf_loss, crf_beam_logits, beam_targets, tgt_key_padding_mask
            elif not grpo_mode and not crf_mode:
                finalized_tokens = self.crf_layer( emissions, labels, masks = emission_mask, grpo_mode = False, crf_mode = False, 
                                                 repeat_penalty = repeat_penalty )
                return finalized_tokens, tgt_key_padding_mask
            elif grpo_mode and crf_mode and use_crf_beam_logits:
                finalized_tokens, beam_probs, log_beam_probs, crf_loss, crf_beam_logits, beam_targets, sampled_beam_idx, b_finalized_tokens = \
                self.crf_layer( emissions, labels, sampled_beam_idx = sampled_beam_idx, masks = emission_mask, grpo_mode = True, 
                                crf_mode = True, use_crf_beam_logits = True, repeat_penalty = repeat_penalty  )
                return finalized_tokens, beam_probs, log_beam_probs, emissions, crf_loss, crf_beam_logits, beam_targets, \
                sampled_beam_idx, b_finalized_tokens, tgt_key_padding_mask
        else:
            finalized_tokens, crf_loss, crf_beam_logits, beam_targets = self.crf_layer( emissions, labels, masks = emission_mask, 
                                                                                    grpo_mode = grpo_mode, crf_mode = crf_mode,
                                                                                    use_crf_beam_logits = use_crf_beam_logits, 
                                                                                        repeat_penalty = repeat_penalty  )
            return finalized_tokens, emissions, crf_loss, crf_beam_logits, beam_targets, predicted_text_logits, tgt_key_padding_mask



In [ ]:
# 各スコアの計算方法の準備
fn = SmoothingFunction().method7
rouge_func = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True) 
bert_score = load('bertscore')
wer_metric = load("wer")
batch_size = 4

def compute_scores( hypo_ids, labels ):

    #print( "hypo_ids:", hypo_ids )
    #print( "labels:", labels )
    #print( "hypo_ids.size():", hypo_ids.size() )
    device = hypo_ids.device
    
    preds1 = hypo_ids.clone().to(device)
    targets1 = labels.clone().to(device)
    
    # 2. リスト内包表記の構文エラーを修正
    preds_decoded = tokenizer.batch_decode(preds1, skip_special_tokens=False)
    preds_str = [s.replace("[PAD]", "").strip() for s in preds_decoded]
    
    # deepcopy を使って完全に複製
    preds_str2 = copy.deepcopy(preds_str)

    targets_decoded = tokenizer.batch_decode(targets1, skip_special_tokens=False)
    targets_str = [s.replace("[PAD]", "").strip() for s in targets_decoded]
    # deepcopy を使って完全に複製
    targets_str2 = copy.deepcopy(targets_str)

    def format_for_er(text_list):
        # 例: "私は学生" -> "私 は 学 生"
        return [" ".join(list(text.replace(" ", ""))) for text in text_list]

    # デコードした文字列を1文字ずつのスペース区切りに変換
    preds_str_spaced = format_for_er(preds_str)
    targets_str_spaced = format_for_er(targets_str)
    
    ## WERを計算
    individual_wer_scores = [
        wer_metric.compute(predictions=[pred], references=[ref]) if ref.strip() else 0.0
        for pred, ref in zip(preds_str_spaced, targets_str_spaced)
    ]
    individual_wer_scores = torch.tensor( individual_wer_scores ).to(device)

    # 3. ROUGEスコアのキーを 'rouge2' に修正 (fmeasureは[2]または.fmeasure)
    rouge1_scores = [[rouge_func.score(target, pred)['rouge1'].fmeasure] for pred, target in zip(preds_str, targets_str)]
    rouge1_scores = torch.tensor(rouge1_scores).to(device).squeeze(-1).float()

    rouge2_scores = [[rouge_func.score(target, pred)['rouge2'].fmeasure] for pred, target in zip(preds_str, targets_str)]
    rouge2_scores = torch.tensor(rouge2_scores).to(device).squeeze(-1).float()

    rougeL_scores = [[rouge_func.score(target, pred)['rougeL'].fmeasure] for pred, target in zip(preds_str, targets_str)]
    rougeL_scores = torch.tensor(rougeL_scores).to(device).squeeze(-1).float()
    
    # 5. BLEUの入力形式をトークン単位・参照文ネスト構造に変換
    # 参照文 (references) は [[['単語', '単語']], [['単語']]] の3次元構造が必要
    tokenized_targets = [[target.split()] for target in targets_str]
    tokenized_preds = [pred.split() for pred in preds_str]
    
    ## 変数名の競合を防ぐためスコアは別名で保存
    #reward_bleu = corpus_bleu(tokenized_targets, tokenized_preds, smoothing_function=fn)        
    # --- 各データごとのBLEUスコアを計算 ---
    # zipで1ペアずつ取り出して sentence_bleu に渡します
    individual_bleu_scores = [
        sentence_bleu(ref, hyp, smoothing_function=fn)
        for ref, hyp in zip(tokenized_targets, tokenized_preds)
    ]
    individual_bleu_scores = torch.tensor( individual_bleu_scores ).to(device).to(torch.float )
    #print( "individual_bleu_scores.dtype:", individual_bleu_scores.dtype )
    #print( "individual_bleu_scores:", individual_bleu_scores )
    
    preds_str2 = [p if p.strip() != "" else "." for p in preds_str2]
    targets_str2 = [t if t.strip() != "" else "." for t in targets_str2]
    
    ## 4. BERTScore の計算
    with torch.no_grad():
        model_name = 'distilbert-base-uncased'
        bert_scores = bert_score.compute(
            predictions=preds_str2,
            references=targets_str2,
            model_type=model_name,
            use_fast_tokenizer=False,
            #use_fast_tokenizer=True,
            lang='en',
            #device=device,
            device = torch.device( "cuda:0" ),
            #batch_size = 1,
            batch_size=batch_size,
            rescale_with_baseline=False
        )['f1']
        
    bert_scores = torch.tensor(bert_scores).to(device)

    return rouge1_scores, rouge2_scores, rougeL_scores, bert_scores, individual_bleu_scores, individual_wer_scores, preds_str, targets_str
    #return rouge1_scores, rouge2_scores, rougeL_scores, bert_scores, individual_wer_scores, preds_str, targets_str


In [ ]:
hypo_ids = torch.randint( 0, len( tokenizer ), size = ( 4, 100 ) )
labels = torch.randint( 0, len( tokenizer), size = ( 4, 100 ) )

rouge1_scores, rouge2_scores, rougeL_scores, bert_scores, individual_bleu_scores, individual_wer_scores, preds_str, targets_str \
 = compute_scores( hypo_ids, labels )

#rouge1_scores, rouge2_scores, rougeL_scores, bert_scores, individual_wer_scores, preds_str, targets_str \
# = compute_scores( hypo_ids, labels )


print( rouge1_scores.size() )
print( rouge2_scores.size() )
print( rougeL_scores.size() )
print( bert_scores.size() )
print( individual_bleu_scores.size() )
print( individual_wer_scores.size() )
print( preds_str )
print( targets_str )

In [ ]:
# 警告（WARNING）レベルのログを非表示にし、エラーのみ表示する
logging.set_verbosity_error()

model_checkpoint = "google-bert/bert-large-uncased"  # または "facebook/bart-base"
max_input_length = 512
max_target_length = 128
batch_size = 40 # VRAMに合わせて調整（4, 8, 16など）
batch_size = 24
is_glat_thresh = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device( "cpu" )
print("Device:", device)
tokenizer = BertTokenizer.from_pretrained(model_checkpoint, model_max_length=512)

# Load from the file
with open("data_bert_train_t128.pkl", "rb") as f:
#with open("data_bert_val_t128.pkl", "rb") as f:
    tokenized_dataset = pickle.load(f)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer )

# 1エポックあたりに読み込むデータ数
#num_samples = 150000 #7時間
#num_samples = 100000 #5時間

# サンプラーを設定（replacement=False で重複なし、num_samples で件数指定）
# ※注意: num_samples を指定する場合は replacement=True にするか、特定の条件が必要です
# 重複なしで制限したい場合は Subset（方法1）のほうが確実です
#sampler = RandomSampler(tokenized_dataset, replacement=True, num_samples=num_samples)

train_dataloader = DataLoader(
    tokenized_dataset,
    shuffle=True,  # 訓練データはシャッフルする
    batch_size=batch_size,
    #sampler = sampler,
    collate_fn=data_collator
)

print( "len( train_dataloader ):", len( train_dataloader ) )


# Load from the file
with open("data_bert_val_t128.pkl", "rb") as f:
    val_tokenized_dataset = pickle.load(f)


# 1エポックあたりに読み込むデータ数
val_num_samples = 1000
#val_num_samples = 300

## サンプラーを設定（replacement=False で重複なし、num_samples で件数指定）
## ※注意: num_samples を指定する場合は replacement=True にするか、特定の条件が必要です
## 重複なしで制限したい場合は Subset（方法1）のほうが確実です
#val_sampler = SequentialSampler(val_tokenized_datase)

# 例: 先頭の1000件だけを検証データとして使いたい場合
limited_val_dataset = Subset(val_tokenized_dataset, range(val_num_samples))

val_dataloader = DataLoader(
    limited_val_dataset,
    shuffle=False,  # 訓練データはシャッフルする
    batch_size=batch_size,
    #sampler = val_sampler, 
    collate_fn=data_collator
)

print( "len( val_dataloader ):", len( val_dataloader ) )

#train_dataloader = val_dataloader

## Load from the file
#with open("data_bert_test_t128.pkl", "rb") as f:
#    test_tokenized_dataset = pickle.load(f)

## 1エポックあたりに読み込むデータ数
#num_samples = 1000
#
## サンプラーを設定（replacement=False で重複なし、num_samples で件数指定）
## ※注意: num_samples を指定する場合は replacement=True にするか、特定の条件が必要です
## 重複なしで制限したい場合は Subset（方法1）のほうが確実です
##sampler = SequentialSampler(test_tokenized_dataset, num_samples=num_samples)
# 1エポックあたりに読み込むデータ数
#test_num_samples = 10000

### サンプラーを設定（replacement=False で重複なし、num_samples で件数指定）
### ※注意: num_samples を指定する場合は replacement=True にするか、特定の条件が必要です
### 重複なしで制限したい場合は Subset（方法1）のほうが確実です
###test_sampler = SequentialSampler(val_tokenized_datase)

## 例: 先頭の1000件だけを検証データとして使いたい場合
#limited_test_dataset = Subset(test_tokenized_dataset, range(test_num_samples))

#
#test_dataloader = DataLoader(
#    limited_test_dataset,
#    shuffle=False,  # 訓練データはシャッフルする
#    batch_size=batch_size,
#    #sampler = sampler, 
#    collate_fn=data_collator
#)

#print( "len( test_dataloader ):", len( test_dataloader ) )

# train データ数28万 batch = 16 → 17500  batch = 64 → 4350 batch = 32 →　8700 batcfh = 40 → 7000
epochs = 10
num_display_train = 50
#num_display_train = 3
train_score_param = len( train_dataloader ) // num_display_train
train_caption_param = len( train_dataloader ) // num_display_train
print( "train_score_pram:", train_score_param )
print( "train_caption_param:", train_caption_param )
#train_score_param = 3
#train_caption_param = 3
num_display_val = 20
#num_display_val = 3
val_param = len( train_dataloader ) // num_display_val
#val_param = 10
#val_param = 3
print( "val_param:", val_param )
all_global_steps = len( train_dataloader ) * epochs
all_val_global_steps = epochs * num_display_val * len( val_dataloader )
lr_bert = 1.0677805054996179e-05
lr_td = 0.0001404696142205931
lr_crf = 0.00011594577401432891
lr_emission = 7.5432814081631705e-06
lr_others = 0.00011993337517254489
clip_grad_threshold = 4.584536582378958
weight_decay = 0.002445456885753439
#crf_coef = 1.0
ce_coef = 0.6488161409440697 * 3
length_coef = 0.529782234041065 * 3 
#repeat_penalty = 0.04553827242960138
repeat_penalty = 0.04
use_tgt_interpolate = 'none'
betas = (0.9, 0.999 )
warmup = 0.1
#全ステップ数 all_global_steps
num_warmup_steps = all_global_steps * warmup
use_amp = True
#use_amp = False
PATH = 'model/model_summarize_5trial_best_backup3.pth'
print( "PATH:", PATH )
print( "exist saved_pth:", os.path.isfile(PATH) ) 
#use_saved_pth = True
use_saved_pth = False
print( "use_saved_pth:", use_saved_pth )

   
model = BertCRF( tokenizer, device = device, top_p = 1.0, 
                 top_k = 0, num_samples = 1, cand = 0, sft_mode = True )        
model.to(device)

# 最適化手法の定義
# Optimizerの生成, clipとそうでないモジュールとの
# パラメータで異なる学習率を適用
params_bert = []
params_td = []
params_crf = []
params_emission = []
params_others = []
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        if 'bert' in name:
            #print( "bert name:", name )
            params_bert.append(parameter)
            #parameter.requires_grad = False
        elif 'transformer_decoder' in name:
            #print( "td name:", name )
            params_td.append(parameter)
            #parameter.requires_grad = False

        elif 'crf_layer' in name:
            #print( "crf_layer name:", name )
            params_crf.append(parameter)
            #parameter.requires_grad = False
        elif 'emission_' in name:
            #print( "emission name:", name )
            params_emission.append(parameter)
        else:
            #print( "others name:", name )
            params_others.append(parameter)
param_groups = [
    {'params': params_bert, 'lr': lr_bert},
    {'params': params_td, 'lr': lr_td},
    {'params': params_crf, 'lr': lr_crf},
    {'params': params_emission, 'lr': lr_emission},
    {'params': params_others, 'lr': lr_others}]
optimizer = torch.optim.AdamW( param_groups, weight_decay = weight_decay, betas=betas )

# 5. オプティマイザの設定（PyTorch標準）
#optimizer = AdamW(model.parameters(), lr=5e-5)
criteria = nn.CrossEntropyLoss( ignore_index = tokenizer.pad_token_id )
criterion_length = nn.CrossEntropyLoss()
scheduler = get_cosine_schedule_with_warmup( optimizer, num_warmup_steps, all_global_steps )   

if use_saved_pth and os.path.isfile(PATH):
    checkpoint = torch.load(PATH, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint['model_state_dict'], strict = False)
    model.to(device)
    print( "model parameters were loaded")
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    ## optimizerのstateを現在のdeviceに移す。これをしないと、保存前後でdeviceの不整合が起こる可能性がある。
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(device)
    loss = checkpoint['loss']
    begin_epoch = checkpoint['epoch']
    global_steps = checkpoint['global_steps']
    val_global_steps = checkpoint['val_global_steps']
    #val_global_steps = int( val_global_steps * ( 1000 / 300 ) )
    #begin_epoch = checkpoint['epoch']
    #global_step = checkpoint['global_step']  
else:
    begin_epoch = 0
    global_steps = 0
    val_global_steps = 0

#for module in model.modules():
#    if isinstance(module, torch.nn.Dropout):
#        module.p = 0.0

# 学習経過の書き込み
now = datetime.datetime.now()
train_loss_file = '{}/MyOriginal_train_loss_{}.csv'\
    .format("model/", now.strftime('%Y%m%d_%H%M%S'))
with open(train_loss_file, 'a') as f:
    print(f'{all_global_steps}', file=f) 
print( "train_loss_file:", train_loss_file )
val_loss_file = '{}/MyOriginal_val_loss_{}.csv'\
    .format("model/", now.strftime('%Y%m%d_%H%M%S'))
with open(val_loss_file, 'a') as f:
    print(f'{all_val_global_steps}', file=f)

# AMP用のスケーラー
scaler = GradScaler(enabled=use_amp)

#各初期値
#global_steps = 0


train_losses = deque()
train_crfs = deque()
train_ces = deque()
train_lengths = deque()
train_rouge1s = deque()
train_rouge2s = deque()
train_rougeLs = deque()
train_berts = deque()
train_bleus = deque()
train_wers = deque()

#val_global_steps = 0
#val_global_steps = 19
val_avg_return_best = float('-inf' )

for epoch in range(begin_epoch, epochs):
    total_loss = 0
    for num_batch, batch in tqdm( enumerate(train_dataloader), total=len(train_dataloader) ):
    #for num_batch, batch in tqdm( enumerate(val_dataloader), total=len(val_dataloader) ):
        
        optimizer.zero_grad()

        progress = (global_steps) / all_global_steps

        if progress < is_glat_thresh:
            p_glat = 0.8 * ( is_glat_thresh - progress ) / is_glat_thresh
        else:
            p_glat = 0

        #if progress > p_is_inference_thresh:
        #    p_is_inference = True
        #else:
        #    p_is_inference = False
        
        # データをGPU/CPUへ転送
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # 3. 予測（Encoderのみ実行）
        model.train()
        with autocast(str(device),enabled=use_amp):
            finalized_tokens, emissions, crf_loss, crf_beam_logits, beam_targets, predicted_text_logits, tgt_key_padding_mask = model( input_ids, attention_mask, 
                                                                         labels, grpo_mode = False, 
                                                                         crf_mode = True, use_crf_beam_logits = False,
                                                                         p_glat  = p_glat, use_tgt_interpolate = use_tgt_interpolate,
                                                                         is_inference = False, repeat_penalty = repeat_penalty  )
            #print( "predicted_text_length:", predicted_text_length )
            ce_loss = criteria( emissions.transpose(1,2), labels )
            a = 1.0 * crf_loss
            b = ce_coef * ce_loss
            target_text_length = torch.sum( torch.ne( labels, pad_token_id).to(torch.int), dim = -1 )
            #print( "target_text_length:", target_text_length )
            # 損失計算（CrossEntropyLoss）
            loss_length = criterion_length(predicted_text_logits, target_text_length)
            c = length_coef * loss_length
            loss = a + b + c
            
        # 逆伝播とパラメータ更新
        #loss.backward()
        #optimizer.step()
        #scheduler.step()
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(\
                model.parameters(),
                clip_grad_threshold)
        # オプティマイザにより，パラメータを更新する
        scaler.step(optimizer)
        scaler.update()            
        scheduler.step()
        
        total_loss += loss.item()

        hypo_ids = finalized_tokens
        hypo_ids[tgt_key_padding_mask==True] = pad_token_id
        
        global_steps += 1

        rouge1, rouge2, rougeL, bert, bleu, wer, preds_str, targets_str = compute_scores( hypo_ids, labels )

        rouge1_mean = rouge1.mean().item()
        rouge2_mean = rouge2.mean().item()
        rougeL_mean = rougeL.mean().item()
        bert_score_mean = bert.mean().item()
        bleu_score_mean = bleu.mean().item()
        wer_mean = wer.mean().item()

        train_losses.append( loss.mean().item() )
        train_crfs.append( a.mean().item() )
        train_ces.append( b.mean().item() )
        train_lengths.append( c.mean().item() )
        train_rouge1s.append( rouge1_mean )
        train_rouge2s.append( rouge2_mean )
        train_rougeLs.append( rougeL_mean )
        train_berts.append( bert_score_mean )
        train_bleus.append( bleu_score_mean )
        train_wers.append( wer_mean )
        if len(train_losses) > 100:
            train_losses.popleft()
            train_crfs.popleft()
            train_ces.popleft()
            train_lengths.popleft()
            train_rouge1s.popleft()
            train_rouge2s.popleft()
            train_rougeLs.popleft()
            train_berts.popleft()
            train_bleus.popleft()
            train_wers.popleft()

        train_loss = torch.Tensor( train_losses ).mean().item()
        train_crf = torch.Tensor( train_crfs ).mean().item()
        train_ce = torch.Tensor( train_ces ).mean().item()
        train_length = torch.Tensor( train_lengths ).mean().item()
        train_rouge1 = torch.Tensor( train_rouge1s ).mean().item()
        train_rouge2 = torch.Tensor( train_rouge2s ).mean().item()
        train_rougeL = torch.Tensor( train_rougeLs ).mean().item()
        train_bert = torch.Tensor( train_berts ).mean().item()
        train_bleu = torch.Tensor( train_bleus ).mean().item()
        train_wer = torch.Tensor( train_wers ).mean().item()
        train_return = 5.0 * train_rouge1 + 5.0 * train_rouge2 + 5.0 * train_rougeL + 1.0 * train_bert + 10.0 * train_bleu - 1.0 * train_wer
        
        
        if num_batch % train_score_param == train_score_param - 1  or num_batch == 0 or num_batch == len( train_dataloader ) - 1:
            print( "\nTrain phase." )
            with open(train_loss_file, 'a') as f:
                print(f'{global_steps / all_global_steps}, {train_loss}, {train_crf}, {train_ce}, {train_length}, {train_rouge1},',
                      f'{train_rouge2}, {train_rougeL}, {train_bert}, {train_bleu}, {train_wer}, {train_return}', file=f)
    
            print( "progress :", (global_steps) / all_global_steps, "loss", train_loss, "crf_loss:", train_crf, "ce_loss:", train_ce, 
                   "length_loss:", train_length, "rouge1:", train_rouge1, "rouge2:", train_rouge2, "rougeL:", train_rougeL, 
                   "bert:",train_bert, "bleu:", train_bleu, "wer:", train_wer, "return:", train_return )
        
        if num_batch % train_caption_param == train_caption_param - 1 or num_batch == 0 or num_batch == len( train_dataloader ) - 1:
        #if num_batch % 20 == 19 or num_batch == 0 or num_batch == len( train_dataloader ) - 1:
            print( "\nTrain phase." )
            print( "hypo:", preds_str[0] )
            print( "target:", targets_str[0] )
    
        if global_steps % val_param == val_param - 1 or global_steps == all_global_steps:
        #if global_steps % 20 == 19 or global_steps == all_global_steps:
    
            model.eval()
            rouge11 = []
            rouge21 = []
            rougeL1 = []
            bert1 = []
            bleu1 = []
            wer1 = []
            for num_batch, batch in tqdm( enumerate(val_dataloader), total=len(val_dataloader) ):
                # データをGPU/CPUへ転送
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                #attention_mask = (~( batch["input_ids"] == tokenizer.pad_token_ids )).to(torch.float).to(device)
                #print( "attntion_mask:",attention_mask )
                labels = batch["labels"].to(device)
        
                # 3. 予測（Encoderのみ実行）
                with torch.no_grad():
                    finalized_tokens, emissions, crf_loss, crf_beam_logits, beam_targets, _, tgt_key_padding_mask = model( input_ids, attention_mask, labels, 
                                                                                    grpo_mode = False, crf_mode = True, 
                                                                                    use_crf_beam_logits = False,
                                                                                    p_glat = 0.0, 
                                                                                    use_tgt_interpolate = use_tgt_interpolate,
                                                                                    is_inference = True,
                                                                                    repeat_penalty = repeat_penalty  )
        
                hypo_ids = finalized_tokens
                hypo_ids[tgt_key_padding_mask==True] = pad_token_id
                
                rouge1, rouge2, rougeL, bert, bleu, wer, preds_str, targets_str = compute_scores( hypo_ids, labels )
        
 
                # this mean is batch mean.
                rouge11.append( rouge1.detach().cpu() )
                rouge21.append( rouge2.detach().cpu() )
                rougeL1.append( rougeL.detach().cpu() )
                bert1.append( bert.detach().cpu() )
                bleu1.append( bleu.detach().cpu() )
                wer1.append( wer.detach().cpu() )
    
                val_global_steps += 1
                
                if num_batch % (len( val_dataloader ) // 3 )== ( len( val_dataloader ) // 3 - 1 ):
                    # this avg is num_batch average.
                    avg_rouge1 = torch.cat( rouge11 ).mean().item()
                    avg_rouge2 = torch.cat( rouge21 ).mean().item()
                    avg_rougeL = torch.cat( rougeL1 ).mean().item()
                    avg_bert = torch.cat( bert1 ).mean().item()
                    avg_bleu = torch.cat( bleu1 ).mean().item()
                    avg_wer = torch.cat( wer1 ).mean().item()
                    avg_return = 5.0 * avg_rouge1 + 5.0 * avg_rouge2 + 5.0 * avg_rougeL + 1.0 * avg_bert + 10.0 * avg_bleu - 1.0 * avg_wer
                    print( "\nVal phase." )
                    print( "progress :", (val_global_steps / all_val_global_steps), "avg_rouge1:", avg_rouge1, "avg_rouge2:", avg_rouge2, 
                           "avg_rougeL:", avg_rougeL,"avg_bert:", avg_bert, "avg_bleu:", avg_bleu, "avg_wer:", avg_wer, 
                           "avg_return", avg_return )
                    print( "hypo:", preds_str[0] )
                    print( "target:", targets_str[0] )
    
            #if num_batch == len( val_dataloader ) - 1:
            # this avg is num_batch average.
            all_rouge1 = torch.cat( rouge11 )
            all_rouge2 = torch.cat( rouge21 )
            all_rougeL = torch.cat( rougeL1 )
            all_bert = torch.cat( bert1 )
            all_bleu = torch.cat( bleu1 )
            all_wer = torch.cat( wer1 )
            avg_rouge1 = all_rouge1.mean().item()
            avg_rouge2 = all_rouge2.mean().item()
            avg_rougeL = all_rougeL.mean().item()
            avg_bert = all_bert.mean().item()
            avg_bleu = all_bleu.mean().item()
            avg_wer = all_wer.mean().item()
            avg_return = 5.0 * avg_rouge1 + 5.0 * avg_rouge2 + 5.0 * avg_rougeL + 1.0 * avg_bert + 10.0 * avg_bleu - 1.0 * avg_wer
           
            with open(val_loss_file, 'a') as f:
                print(f'{(val_global_steps / all_val_global_steps)}, {avg_rouge1}, {avg_rouge2}, {avg_rougeL},',
                f'{avg_bert}, {avg_bleu}, {avg_wer}, {avg_return}', file=f)
            if val_avg_return_best < avg_return:
                # モデルを保存
                torch.save({'epoch': epoch, 
                    'global_steps': global_steps, 
                    'val_global_steps': val_global_steps,
                    'loss': loss,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    },
                    f'model/model_summarize_15_5trial_best.pth')
                val_avg_return_best = avg_return
                               # 4. 中間報告 (枝刈り用)
            

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")


In [ ]:
'''
torch.save({'val_global_steps': val_global_steps,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    #'scheduler_state_dict': scheduler.state_dict(),
    'loss': loss,
    },
    f'model/model_summarize_4_0_best.pth')
'''